In [1]:
%%time
import warnings
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from catboost import CatBoostRegressor, Pool, EShapCalcType, EFeaturesSelectionAlgorithm
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings('ignore')

Wall time: 5min 36s


In [18]:
df = pd.read_csv("data/Train.csv")

df.drop(["ID"], axis=1, inplace=True)

In [19]:
(df.isna().sum()/df.shape[0]).sort_values(ascending=False)

2appDaysUrea                          0.697674
2tdUrea                               0.696124
CropOrgFYM                            0.690956
Ganaura                               0.624548
BasalUrea                             0.440310
PCropSolidOrgFertAppMethod            0.345478
OrgFertilizers                        0.344961
TransIrriCost                         0.227907
1appDaysUrea                          0.143669
1tdUrea                               0.143669
BasalDAP                              0.140310
TransplantingIrrigationPowerSource    0.129974
FirstTopDressFert                     0.125323
MineralFertAppMethod.1                0.124289
TransDetFactor                        0.074677
SeedlingsPerPit                       0.074677
NursDetFactor                         0.074677
Harv_hand_rent                        0.065116
StandingWater                         0.061499
TransplantingIrrigationHours          0.049871
CropbasalFerts                        0.048579
Transplanting

In [20]:
# features_select=['Block',
#   'CultLand',
#   'CropCultLand',
#   'CropTillageDate',
#   'CropTillageDepth',
#   'CropEstMethod',
#   'RcNursEstDate',
#   'SeedingSowingTransplanting',
#   'NursDetFactor',
#   'TransDetFactor',
#   'TransplantingIrrigationHours',
#   'TransplantingIrrigationSource',
#   'TransplantingIrrigationPowerSource',
#   'StandingWater',
#   'BasalDAP',
#   'FirstTopDressFert',
#   '1tdUrea',
#   'Harv_method',
#   'Harv_date',
#   'Residue_length',
#   'Residue_perc',
#   'Stubble_use',
#   'Acre',
#   'SeedingSowingTransplantingMonth',
#   'CropTillageDate-RcNursEstDate',
#   'Harv_date-Threshing_date',
#   'RcNursEstDate-Threshing_date',
#   'CropTillageDate-Harv_date',
#   'RcNursEstDate-SeedingSowingTransplanting',
#   'Harv_date-SeedingSowingTransplanting',
#   'Yield']

In [21]:
### HANDLING MISSING DATA ###

cat_features = df.select_dtypes(np.object_).columns.to_list()
num_features = df.select_dtypes(np.number).columns.to_list()
num_features.remove("Yield")

df[cat_features]=df[cat_features].fillna(df.mode().iloc[0])
df[num_features]=df[num_features].fillna(df.mean())


# df = df[df.columns[df.isna().sum()/df.shape[0] < 0.2 ]]

In [22]:
def create_date_features(data, feature):
    return pd.DatetimeIndex(data[feature]).month.map(str)


def create_duration_features(data, features):
    duration_df = pd.DataFrame()
    data[features]=data[features].apply(pd.to_datetime)
    patterns=[i.split("/") for i in (set(["/".join(sorted(list(set([i,j])))) for i in features for j in features if len(set([i,j]))>1]))]
    for i in patterns:
        duration_df[f"{i[0]}-{i[1]}"]=(data[i[1]]-data[i[0]]).dt.days
    return duration_df

In [23]:
features=["CropTillageDate","RcNursEstDate","SeedingSowingTransplanting","Harv_date","Threshing_date"]

df["CropTillageMonth"]=create_date_features(df, "CropTillageDate")
df["RcNursEstMonth"]=create_date_features(df, "RcNursEstDate")
df["SeedingSowingTransplantingMonth"]=create_date_features(df,"SeedingSowingTransplanting")
df["Harv_Month"]=create_date_features(df, "Harv_date")
df["Threshing_Month"]=create_date_features(df, "Threshing_date")

duration_df=create_duration_features(df.copy(), features)

df = pd.concat([df, duration_df], axis=1)


In [24]:
#df = df[df.select_dtypes(np.number).columns.to_list()]
#df.dropna(axis=0, inplace=True)

In [25]:
X = df.drop("Yield", axis=1)
y = df.Yield 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1234)

In [178]:
train_data=Pool(
    data=X_train,
    label=y_train,
    cat_features=X.select_dtypes(np.object_).columns.to_list()
)
eval_data=Pool(
    data=X_test,
    label=y_test,
    cat_features=X.select_dtypes(np.object_).columns.to_list()
)

In [179]:
X.shape

(2157, 30)

In [106]:
model = CatBoostRegressor(iterations=1000, loss_function='RMSE',random_seed=0)
summary = model.select_features(
    train_data,
    eval_set=eval_data,
    features_for_select='0-48',
    num_features_to_select=30,
    steps=15,
    algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
    shap_calc_type=EShapCalcType.Regular,
    train_final_model=True,
    logging_level='Silent',
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [107]:
# loss=summary["loss_graph"]["loss_values"]
# eliminated_features=summary["eliminated_features_names"]
# features=[]
# for i,feature in enumerate(eliminated_features):
#     if (loss[i]<loss[i+1]):
#         features.append(feature)

summary["selected_features_names"]

['Block',
 'CultLand',
 'CropCultLand',
 'CropTillageDate',
 'CropTillageDepth',
 'CropEstMethod',
 'RcNursEstDate',
 'SeedingSowingTransplanting',
 'NursDetFactor',
 'TransDetFactor',
 'TransplantingIrrigationHours',
 'TransplantingIrrigationSource',
 'TransplantingIrrigationPowerSource',
 'StandingWater',
 'BasalDAP',
 'FirstTopDressFert',
 '1tdUrea',
 'Harv_method',
 'Harv_date',
 'Residue_length',
 'Residue_perc',
 'Stubble_use',
 'Acre',
 'SeedingSowingTransplantingMonth',
 'CropTillageDate-RcNursEstDate',
 'Harv_date-Threshing_date',
 'RcNursEstDate-Threshing_date',
 'CropTillageDate-Harv_date',
 'RcNursEstDate-SeedingSowingTransplanting',
 'Harv_date-SeedingSowingTransplanting']

## Modelisation

In [26]:
cv=KFold(n_splits=5,shuffle=True,random_state=0)

In [27]:
model=CatBoostRegressor(iterations=1000,learning_rate=0.05,depth=8)

scores=[]
for i,j in enumerate(cv.split(X,y)):
    train=Pool(
        data=X.iloc[j[0]],
        label=y.iloc[j[0]],
        cat_features=X.select_dtypes(np.object_).columns.to_list()
    )
    eval=Pool(
        data=X.iloc[j[1]],
        label=y.iloc[j[1]],
        cat_features=X.select_dtypes(np.object_).columns.to_list()
    )
    model.fit(
        train,
        eval_set=eval,
        verbose=False
    )
    y_pred=model.predict(X.iloc[j[1]])
    score=mean_squared_error(y.iloc[j[1]], y_pred, squared=False)  
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:364.0739982186916

Fold 1	:773.9120751266338

Fold 2	:450.3541247974215

Fold 3	:318.3443462042192

Fold 4	:188.90930906653176

CV_SCORES_mean: 419.1187706826996
CV_SCORES_std: 196.48701944256138


In [185]:
train=Pool(
    data=X_train,
    label=y_train,
    cat_features=X.select_dtypes(np.object_).columns.to_list()
)
eval=Pool(
    data=X_test,
    label=y_test,
    cat_features=X.select_dtypes(np.object_).columns.to_list()
)
model.fit(
    train,
    eval_set=eval,
    verbose=False
)

## Submission

In [188]:
sub = pd.read_csv("data/SampleSubmission.csv")
test = pd.read_csv("data/Test.csv")
test.drop(["ID"], axis=1, inplace=True)

In [189]:
test["CropTillageMonth"]=create_date_features(test, "CropTillageDate")
test["RcNursEstMonth"]=create_date_features(test, "RcNursEstDate")
test["SeedingSowingTransplantingMonth"]=create_date_features(test,"SeedingSowingTransplanting")
test["Harv_Month"]=create_date_features(test, "Harv_date")
test["Threshing_Month"]=create_date_features(test, "Threshing_date")

duration_test_df=create_duration_features(test.copy(), features)

test = pd.concat([test, duration_test_df], axis=1)

test[cat_features]=test[cat_features].fillna(test.mode().iloc[0])
test[num_features]=test[num_features].fillna(test.mean())

test = test[features_select]

In [190]:
test.shape

(1290, 30)

In [191]:
predict=model.predict(test)

In [192]:
sub.Yield = predict
sub.head()

,ID,Yield
0,ID_F9XXEXN2ADR2,653.182186
1,ID_SO3VW2X4QO93,436.961399
2,ID_UKUQ7JM8E894,504.481328
3,ID_QUISMWEZR2H4,394.462183
4,ID_25JGI455VKCZ,626.022928


In [193]:
sub.to_csv('submission_2.csv', index = False)